## Part of the TauREx retrieval tutorial by Mael Voyer
## This notebook requires a working pyhton envirenment with matplotlib, taurex, and numpy.

In [ ]:
import matplotlib.pyplot as plt
from taurex.parameter import ParameterParser
# optionnal for notebooks with ipympl
#%matplotlib widget
plt.style.use('default')

In [ ]:
pp_path = 'your/path/to/par/file.par' #Path to the parameter file.

pp = ParameterParser() #Creating an ampty parameter parser class
pp.read(pp_path) #Reading the parameter file and filling the parameter parser class
pp.setup_globals() # Setting up the global parameters (e.g. planet mass, radius, etc)
model = pp.generate_appropriate_model() #Generating the appropriate model (e.g. transmission, emission, etc) based on the parameter file
model.build() # Building the model (e.g. creating the atmosphere, setting up the chemistry, etc)

wavenumber_grid ,  flux, _, _ = model.model() #Computing the forward model and retrieving the wavenumber grid and the flux (or transit depth) of the forward model
wavenumber_grid_contrib, contrib_dic = model.model_contrib() #Computing the contribution of each process (e.g. absorption, scattering, etc) to the forward model and retrieving the wavenumber grid and the contribution dictionary
wavenumber_grid_contrib_full, contrib_dic_full = model.model_full_contrib() #Computing the all the sub contributions e.g., each molecule in the absorption contribution

In [ ]:
plt.figure(figsize=(10, 6), tight_layout=True)

for key in contrib_dic.keys(): #Looping over the keys of the contribution dictionary (e.g. Absorption, Scattering, etc) and plotting the contribution of each process to the forward model
    if key == 'Absorption':
        plt.plot(1e4/wavenumber_grid, contrib_dic[key][0], label=key, alpha=0.5)
    else:
        plt.plot(1e4/wavenumber_grid, contrib_dic[key][0], label=key, alpha=0.8, ls = ':', lw = 1)
        
for key in contrib_dic_full.keys():
    if key == 'Absorption':
        for mol in contrib_dic_full[key]: #Plotting the moelcule by molecule contribution to the absorption contribution
            plt.plot(1e4/wavenumber_grid, mol[1], label=mol[0], alpha=0.5)

plt.plot(1e4/wavenumber_grid, flux, label='Forward model', ls = '-', lw = 0.1, color = 'k') #Plotting the full forward model
plt.xlabel('Wavelength (μm)')
plt.ylabel('Transit Depth or flux')
plt.xscale('log')
plt.legend(ncol=2, frameon=False, fontsize=10)
plt.show()

As the contribution are plotted at high resolution the above plot is not very readible. Let us bin them down. We can first create a custom wavelength grid a a resolution of our choosing.

In [ ]:
from taurex.binning import FluxBinner
import numpy as np

def create_grid_res(resolution, wave_min, wave_max): # A function to create a grid with a given resolution and wavelength range
    wave_list = []
    width_list = []
    wave = wave_min
    width = wave/resolution    
    
    while wave < wave_max:
        width = wave / (resolution - 0.5) + width/2/(resolution - 0.5)
        wave = resolution * width 
        width_list.append(width)
        wave_list.append(wave)

    return np.array((wave_list ,width_list)).T

In [ ]:
resolution = 300
binned_wavelength_grid = create_grid_res(resolution, 0.3, 10)[:,0] #Creating a wavelength grid with a resolution of 300 and a wavelength range of 0.3 to 30 microns
binner = FluxBinner(binned_wavelength_grid) #Creating a flux binner class with the binned wavelength grid

Another option is to use the observation wavelength grid :

In [ ]:
obs_wave, obs_flux, obs_err, obs_width = np.loadtxt('your/path/to/observation/file.dat', unpack=True) #Loading the observed spectrum 
binner = FluxBinner(obs_wave) #Creating a flux binner class with the observed wavelength grid

In [ ]:
plt.figure(figsize=(10, 6), tight_layout=True)

binned_wavelength, binned_flux, _, _ = binner.bindown(1e4/wavenumber_grid, flux) #Binning the flux of the forward model to the binned wavelength grid
plt.plot(binned_wavelength, binned_flux, label='Binned Forward model', ls = '-', lw = 1, color = 'k') #Plotting the binned forward model

for key in contrib_dic.keys(): #Looping over the keys of the contribution dictionary (e.g. Absorption, Scattering, etc) and plotting the contribution of each process to the forward model
    binned_wave, binned_flux_contrib, _, _ = binner.bindown(1e4/wavenumber_grid, contrib_dic[key][0]) #Binning the contribution of each process to the forward model to the binned wavelength grid
    if key == 'Absorption':
        plt.plot(binned_wave, binned_flux_contrib, label=key, alpha=0.5)
    else:
        plt.plot(binned_wave, binned_flux_contrib, label=key, alpha=0.8, ls = ':', lw = 1)
        
for key in contrib_dic_full.keys():
    if key == 'Absorption':
        for mol in contrib_dic_full[key]: #Plotting the moelcule by molecule contribution to the absorption contribution
            binned_wave, binned_flux_full_contrib, _, _ = binner.bindown(1e4/wavenumber_grid, mol[1]) #Binning the contribution of each process to the forward model to the binned wavelength grid
            plt.plot(binned_wave, binned_flux_full_contrib, label=mol[0], alpha=0.5)

plt.xlabel('Wavelength (μm)')
plt.ylabel('Transit Depth or flux')
plt.xscale('log')
plt.legend(ncol=2, frameon=False, fontsize=12)
plt.show()
